In [ ]:
import unicodedata
import string
import re
import random
import time
import math
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

# =============================================================================
# 1. Configuration
# =============================================================================
USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")

SOS_token = 0
EOS_token = 1
MAX_LENGTH = 10

# hyperparmeters
hidden_size = 500
n_layers = 2
dropout_p = 0.05
learning_rate = 0.0001
n_epochs = 50000
plot_every = 200
print_every = 1000
teacher_forcing_ratio = 0.5
clip = 5.0


# =============================================================================
# 2. Data Processing
# =============================================================================
class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def index_words(self, sentence):
        if self.name == "cn":
            for word in sentence:
                self.index_word(word)
        else:
            for word in sentence.split(" "):
                self.index_word(word)

    def index_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


def unicode_to_ascii(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )


def normalize_string(s):
    s = unicode_to_ascii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z\u4e00-\u9fa5.!?，。？]+", r" ", s)
    return s


def read_langs(lang1, lang2, reverse=False):
    print("Reading lines...")
    lines = (
        open("%s-%s.txt" % (lang1, lang2), encoding="utf-8").read().strip().split("\n")
    )
    pairs = [[normalize_string(s) for s in line.split("\t")] for line in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs


def filter_pair(p):
    return len(p[1].split(" ")) < MAX_LENGTH


def filter_pairs(pairs):
    return [pair for pair in pairs if filter_pair(pair)]


def prepare_data(lang1_name, lang2_name, reverse=False):
    input_lang, output_lang, pairs = read_langs(lang1_name, lang2_name, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filter_pairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Indexing words...")
    for pair in pairs:
        input_lang.index_words(pair[0])
        output_lang.index_words(pair[1])
    return input_lang, output_lang, pairs


def indexes_from_sentence(lang, sentence):
    if lang.name == "cn":
        return [lang.word2index[word] for word in sentence]
    else:
        return [lang.word2index[word] for word in sentence.split(" ")]


def variable_from_sentence(lang, sentence):
    indexes = indexes_from_sentence(lang, sentence)
    indexes.append(EOS_token)
    var = torch.LongTensor(indexes).view(-1, 1)
    return var.to(device)


def variables_from_pair(input_lang, output_lang, pair):
    input_variable = variable_from_sentence(input_lang, pair[0])
    target_variable = variable_from_sentence(output_lang, pair[1])
    return (input_variable, target_variable)


def as_minutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return "%dm %ds" % (m, s)


def time_since(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return "%s (- %s)" % (as_minutes(s), as_minutes(rs))


# =============================================================================
# 3. Model Definitions
# =============================================================================
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers=1):
        super(EncoderRNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, n_layers)

    def forward(self, word_inputs, hidden):
        seq_len = len(word_inputs)
        embedded = self.embedding(word_inputs).view(seq_len, 1, -1)
        output, hidden = self.rnn(embedded, hidden)
        return output, hidden

    def init_hidden(self):
        return torch.zeros(self.n_layers, 1, self.hidden_size, device=device)


class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, n_layers=1, dropout_p=0.1):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout_p = dropout_p
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, n_layers, dropout=dropout_p)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, word_input, last_hidden):
        word_embedded = self.embedding(word_input).view(1, 1, -1)
        rnn_output, hidden = self.rnn(word_embedded, last_hidden)
        rnn_output = rnn_output.squeeze(0)
        output = F.log_softmax(self.out(rnn_output), dim=1)
        return output, hidden


# =============================================================================
# 4. Training Logic
# =============================================================================
def train_step(
    input_variable,
    target_variable,
    encoder,
    decoder,
    encoder_optimizer,
    decoder_optimizer,
    criterion,
    max_length=MAX_LENGTH,
):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    loss = 0

    target_length = target_variable.size()[0]

    encoder_hidden = encoder.init_hidden()
    encoder_outputs, encoder_hidden = encoder(input_variable, encoder_hidden)

    decoder_input = torch.LongTensor([[SOS_token]]).to(device)
    decoder_hidden = encoder_hidden

    use_teacher_forcing = random.random() < teacher_forcing_ratio

    if use_teacher_forcing:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_variable[di])
            decoder_input = target_variable[di]
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_variable[di])
            topv, topi = decoder_output.data.topk(1)
            ni = topi[0][0]
            decoder_input = torch.LongTensor([[ni]]).to(device)
            if ni == EOS_token:
                break

    loss.backward()
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), clip)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), clip)
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length


def run_training(encoder, decoder, input_lang, output_lang, pairs, n_epochs=n_epochs):
    print(f"Starting Training for {n_epochs} epochs with AdamW & Scheduler...")
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    initial_lr = 0.001
    weight_decay = 0.01

    encoder_optimizer = optim.AdamW(
        encoder.parameters(), lr=initial_lr, weight_decay=weight_decay
    )
    decoder_optimizer = optim.AdamW(
        decoder.parameters(), lr=initial_lr, weight_decay=weight_decay
    )

    scheduler_enc = optim.lr_scheduler.CosineAnnealingLR(
        encoder_optimizer, T_max=n_epochs, eta_min=1e-5
    )
    scheduler_dec = optim.lr_scheduler.CosineAnnealingLR(
        decoder_optimizer, T_max=n_epochs, eta_min=1e-5
    )

    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        training_pair = variables_from_pair(
            input_lang, output_lang, random.choice(pairs)
        )
        input_variable = training_pair[0]
        target_variable = training_pair[1]

        loss = train_step(
            input_variable,
            target_variable,
            encoder,
            decoder,
            encoder_optimizer,
            decoder_optimizer,
            criterion,
        )

        scheduler_enc.step()
        scheduler_dec.step()

        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            current_lr = scheduler_enc.get_last_lr()[0]
            print(
                "%s (%d %d%%) %.4f | LR: %.6f"
                % (
                    time_since(start, epoch / n_epochs),
                    epoch,
                    epoch / n_epochs * 100,
                    print_loss_avg,
                    current_lr,
                )
            )

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    return plot_losses


# =============================================================================
# 5. Evaluation & Testing
# =============================================================================
def evaluate(
    encoder, decoder, input_lang, output_lang, sentence, max_length=MAX_LENGTH
):
    with torch.no_grad():
        input_variable = variable_from_sentence(input_lang, sentence)
        encoder_hidden = encoder.init_hidden()
        encoder_outputs, encoder_hidden = encoder(input_variable, encoder_hidden)

        decoder_input = torch.LongTensor([[SOS_token]]).to(device)
        decoder_hidden = encoder_hidden

        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            ni = topi[0][0]
            if ni == EOS_token:
                decoded_words.append("<EOS>")
                break
            else:
                decoded_words.append(output_lang.index2word[ni.item()])

            decoder_input = torch.LongTensor([[ni]]).to(device)

        return decoded_words


def save_models(encoder, decoder):
    print("Saving models...")
    torch.save(encoder.state_dict(), "encoder.pth")
    torch.save(decoder.state_dict(), "decoder.pth")
    print("Models saved to 'encoder.pth' and 'decoder.pth'")


def predict_test_file(filename, encoder, decoder, input_lang, output_lang):
    """
    Read the test.txt, write all the results to test_results.txt, and print 5 randomly to the console.
    """
    print(f"\n--- Processing Test File: {filename} ---")

    if not os.path.exists(filename):
        print(f"Error: {filename} not found.")
        return

    with open(filename, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")

    lines = [line for line in lines if line.strip()]
    total_lines = len(lines)

    num_samples = 5
    sample_indices = set(
        random.sample(range(total_lines), min(total_lines, num_samples))
    )

    output_filename = "test_results.txt"
    print(f"Total lines: {total_lines}")
    print(f"Saving ALL results to: {output_filename}")
    print(f"Printing {len(sample_indices)} random samples to console:\n")

    with open(output_filename, "w", encoding="utf-8") as out_f:
        for i, line in enumerate(lines):
            sentence = normalize_string(line)

            try:
                output_words = evaluate(
                    encoder, decoder, input_lang, output_lang, sentence
                )

                if output_words and output_words[-1] == "<EOS>":
                    output_sentence = " ".join(output_words[:-1])
                else:
                    output_sentence = " ".join(output_words)

                out_f.write(f"{line.strip()}\t{output_sentence}\n")

                if i in sample_indices:
                    print(f"[{i}] Src: {line.strip()}")
                    print(f"    Out: {output_sentence}")
                    print("-" * 30)

            except KeyError as e:
                err_msg = "Error: Unknown word"
                out_f.write(f"{line.strip()}\t{err_msg}\n")
                if i in sample_indices:
                    print(f"[{i}] Src: {line.strip()}")
                    print(f"    Err: Contains unknown word -> {e}")
                    print("-" * 30)

    print(f"\nTranslation finished. All results saved to {output_filename}")


# =============================================================================
# 6. Main
# =============================================================================
if __name__ == "__main__":
    if os.path.exists("cn-eng.txt"):
        input_lang, output_lang, pairs = prepare_data("cn", "eng", False)
        print(f"Sample pair from training data: {random.choice(pairs)}")

        encoder = EncoderRNN(input_lang.n_words, hidden_size, n_layers).to(device)
        decoder = DecoderRNN(
            hidden_size, output_lang.n_words, n_layers, dropout_p=dropout_p
        ).to(device)

        plot_losses = run_training(encoder, decoder, input_lang, output_lang, pairs)

        try:
            plt.figure()
            fig, ax = plt.subplots()
            loc = ticker.MultipleLocator(base=0.2)
            ax.yaxis.set_major_locator(loc)
            plt.plot(plot_losses)
            plt.savefig("training_loss.png")
            print("Loss plot saved as training_loss.png")
        except ImportError:
            pass

        save_models(encoder, decoder)

        predict_test_file("test.txt", encoder, decoder, input_lang, output_lang)

    else:
        print("Error: Dataset 'cn-eng.txt' not found. Please ensure the file exists.")